**Legacy research notebook.** Start with `VoiceNO_Baseline.ipynb` for the current measured GPU workflow. This notebook retains experimental features and unverified prototype claims.


# 🎙️ VoiceNO — Silent Speech Recognition & Neural Disambiguation

**VoiceNO** enables silent, camera-only speech input on Google Colab GPU.

This master notebook runs the complete VoiceNO pipeline:
1. **Pretrained Auto-AVSR Conformer (GPU)**: Visual lip-reading backbone.
2. **In-Browser WebRTC Webcam Recorder**: Push-to-talk recording directly from your laptop camera into Colab.
3. **N-Best Beam Lattice Extraction**: Extracts visual phoneme candidates.
4. **Probabilistic Semantic Rescoring**: Disambiguates visually identical visemes ( -> *park at the mall*).
5. **Personalized LoRA Calibration**: Fine-tune a lightweight personal adapter in ~20 seconds on free T4 GPU.

---
### ⚡ Step 0: Ensure GPU is Enabled
Go to: **Runtime** -> **Change runtime type** -> Select **T4 GPU** -> **Save**.

## 1. Verify GPU Hardware
Check that NVIDIA CUDA GPU is active.

In [ ]:
!nvidia-smi


## 2. Clone VoiceNO Repository & Install Dependencies

In [ ]:
# Upload voiceno_eval_source.zip generated with experiments/package_eval_source.py.
from google.colab import files
from pathlib import Path
import io
import os
import sys
import subprocess
import zipfile

repo_dir = Path('/content/VoiceNO')
repo_dir.mkdir(exist_ok=True)
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Select only voiceno_eval_source.zip')
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    destination = repo_dir.resolve()
    if sum(item.file_size for item in archive.infolist()) > 50 * 1024 * 1024:
        raise ValueError('Source archive is too large')
    for item in archive.infolist():
        if not (destination / item.filename).resolve().is_relative_to(destination):
            raise ValueError('Invalid archive path')
    archive.extractall(destination)
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
del uploaded
print('VoiceNO source ready:', repo_dir)


## 3. Download Auto-AVSR Base Model Checkpoint (~1.0 GB)
Downloads the 3,291-hour pretrained Visual Speech Recognition foundation weights.

In [ ]:
import os
from pathlib import Path

os.makedirs("checkpoints", exist_ok=True)
ckpt_path = Path("checkpoints/vsr_trlrs2lrs3vox2avsp_base.pth")

if not ckpt_path.is_file():
    print("Downloading Auto-AVSR foundation weights...")
    !gdown 1r1kx7l9sWnDOCnaFHIGvOtzuhFyFA88_ -O checkpoints/vsr_trlrs2lrs3vox2avsp_base.pth

print(f"Checkpoint ready at: {ckpt_path} (Size: {os.path.getsize(ckpt_path) / 1e6:.1f} MB)")


## 4. Initialize VoiceNO Pipeline on GPU
Loads the Conformer model and the probabilistic semantic rescorer.

In [ ]:
import cv2
import numpy as np
import torch
import torchvision.io

# 1. OpenCV drop-in replacement for torchvision.io.read_video
def _read_video_cv2(filename, *args, **kwargs):
    cap = cv2.VideoCapture(filename)
    frames = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    if not frames:
        raise ValueError(f"Could not read frames from: {filename}")
    return torch.from_numpy(np.array(frames, dtype=np.uint8)), None, None

torchvision.io.read_video = _read_video_cv2

from voiceno.config import ModelConfig
from voiceno.models.auto_avsr import AutoAVSRModel
from voiceno.context.rescorer import ContextRescorer

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Loading VoiceNO on device: {device}...")

config = ModelConfig(device=device, beam_size=40)
model = AutoAVSRModel(config=config)
model.load_checkpoint("checkpoints/vsr_trlrs2lrs3vox2avsp_base.pth")

rescorer = ContextRescorer()

print("MODEL & RESCORER READY!")


## 5. Live In-Browser Webcam Recorder (Push-to-Talk)
Because Colab runs in the cloud, this cell embeds a camera interface in your browser:
1. Run the cell below.
2. Click **🔴 Start Recording** and silently mouth your sentence (e.g. *"I want to park at the mall"* or *"I want to buy a kite"*).
3. Click **⏹️ Stop & Transcribe**.
4. VoiceNO processes the mouth movements and displays the N-best candidate lattice and the final disambiguated sentence!

In [ ]:
import os
import sys

# 0. Ensure required libraries are installed in this runtime
try:
    import mediapipe
    import av
    import sentencepiece
except ImportError:
    print("Installing runtime dependencies (mediapipe, av, sentencepiece)...")
    !pip install -q av mediapipe opencv-python sentencepiece jiwer gdown

# 1. Ensure working directory is VoiceNO repository
repo_dir = "/content/VoiceNO" if os.path.exists("/content/VoiceNO") else "."
%cd {repo_dir}
!git pull origin main

if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

# 2. Flush Python module cache so latest git code is reloaded
for mod in list(sys.modules):
    if mod.startswith("app") or mod.startswith("voiceno"):
        del sys.modules[mod]

# 3. Ensure Auto-AVSR model checkpoint exists (~1.0 GB)
os.makedirs("checkpoints", exist_ok=True)
ckpt_path = "checkpoints/vsr_trlrs2lrs3vox2avsp_base.pth"
if not os.path.isfile(ckpt_path):
    print("Downloading Auto-AVSR weights (~1.0 GB)...")
    import subprocess
    subprocess.run(["gdown", "1r1kx7l9sWnDOCnaFHIGvOtzuhFyFA88_", "-O", ckpt_path], check=True)

# 4. Initialize model and rescorer if not already loaded in memory
if "model" not in globals() or model is None:
    import cv2
    import numpy as np
    import torch
    import torchvision.io

    def _read_video_cv2(f, *a, **k):
        c = cv2.VideoCapture(f)
        frames = []
        while c.isOpened():
            r, fr = c.read()
            if not r:
                break
            frames.append(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
        c.release()
        if not frames:
            raise ValueError(f"Could not read frames from: {f}")
        return torch.from_numpy(np.array(frames, dtype=np.uint8)), None, None

    torchvision.io.read_video = _read_video_cv2

    from voiceno.config import ModelConfig
    from voiceno.models.auto_avsr import AutoAVSRModel
    from voiceno.context.rescorer import ContextRescorer

    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    print(f"Loading VoiceNO pipeline on {device}...")
    model = AutoAVSRModel(config=ModelConfig(device=device, beam_size=40))
    model.load_checkpoint(ckpt_path)
    rescorer = ContextRescorer()
    print("VoiceNO Model and Rescorer ready!")

# 5. Launch In-Browser Push-to-Talk Webcam Recorder & Pipeline
from app.colab_recorder import run_pipeline
result = run_pipeline(model, rescorer)


## 6. (Optional) Run 1-Click LoRA Personal Calibration (~20s on GPU)
Mouth 8 guided calibration sentences to adapt the model to your personal facial anatomy and lip movements.

In [ ]:
from voiceno.calibration.trainer import PersonalizationTrainer

print("Starting LoRA Attention Personalization on GPU...")
trainer = PersonalizationTrainer(device=device, strategy="conformer_lora")

metrics = trainer.train(user="default", epochs=10, lr=1e-3)
best_loss = metrics.get("best_loss", 0.0)
print(f"Personalization complete! Best loss: {best_loss:.4f}")
print("Personalized profile saved in user_profiles/default/lora.pt")


## 7. Launch Interactive VoiceNO Web UI
Launches a web interface with a shareable public link.

In [ ]:
!python -m voiceno ui --share
